# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.63704306  0.06335813  0.95865427 -0.65764853 -0.95431936]
 [-0.75676882 -0.27671224  0.29062795 -0.58659781 -0.64856117]
 [ 0.14301253 -0.15100735 -0.78169924  0.06490496 -0.38080898]
 [-0.5084261  -0.48576085 -0.04325279  0.96126141  0.3974962 ]
 [-0.53574968  0.32208353  0.46402414 -0.00109237  0.05016288]
 [ 0.0429904  -0.36237325  0.78950109  0.3750101   0.00588264]
 [ 0.23549317 -0.13721691  0.73956854  0.62030758  0.99147007]
 [ 0.25496336 -0.84811114 -0.5652131  -0.81173682 -0.21178536]
 [-0.86023392  0.02260506 -0.4556842  -0.57077542  0.94459184]
 [ 0.91820094  0.69402185 -0.86091338  0.52840228 -0.54820615]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a2', 'a2', 'a1', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 1, 1, 1, 0, 0, 1, 1, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:00<00:29,  1.13it/s]

SVI:   3%|▎         | 1/34 [00:00<00:29,  1.13it/s, loss=2080.5793]

SVI:   6%|▌         | 2/34 [00:00<00:28,  1.13it/s, loss=2487.5723]

SVI:   9%|▉         | 3/34 [00:00<00:27,  1.13it/s, loss=2117.4675]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.22it/s, loss=2528.8689]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.22it/s, loss=1756.8243]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.22it/s, loss=2001.7728]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.54it/s, loss=2001.7728]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.54it/s, loss=2294.6047]